<a href="https://colab.research.google.com/github/ujwal99999/BIS-LAB-5M/blob/main/Resource_allocation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# 1. DEFINE THE RESOURCE ALLOCATION PROBLEM
NUM_TASKS = 5
TOTAL_CAPACITY = 100.0  # Max total resources available across all tasks
MIN_RES = np.array([5, 5, 5, 5, 5])    # Minimum allocation required per task
MAX_RES = np.array([40, 40, 40, 40, 40]) # Maximum allocation allowed per task

# Simulated task metrics (e.g., target demand and priority weights)
IDEAL_DEMAND = np.array([12, 45, 30, 15, 25])
PRIORITY_WEIGHT = np.array([1.5, 3.0, 2.0, 1.0, 2.5])

def evaluate_fitness(x):
    """
    Objective Function: Minimize the weighted squared difference from the ideal demand.
    Includes a harsh mathematical penalty if the total capacity is breached.
    """
    # Calculate penalty if total allocated exceeds what we have
    total_allocated = np.sum(x)
    penalty = 0
    if total_allocated > TOTAL_CAPACITY:
        penalty = 10000 * (total_allocated - TOTAL_CAPACITY) ** 2

    # Cost function (Lower score = better allocation)
    allocation_cost = np.sum(PRIORITY_WEIGHT * (x - IDEAL_DEMAND) ** 2)

    return allocation_cost + penalty

# 2. PSO PARAMETERS
SWARM_SIZE = 30
MAX_ITERATIONS = 50
W = 0.5   # Inertia weight
C1 = 1.5  # Cognitive coefficient (personal best pull)
C2 = 1.5  # Social coefficient (global best pull)

# 3. INITIALIZATION
# Position vector X (swarm_size, num_tasks) and Velocity vector V
X = np.random.uniform(MIN_RES, MAX_RES, size=(SWARM_SIZE, NUM_TASKS))
V = np.random.uniform(-1, 1, size=(SWARM_SIZE, NUM_TASKS))

# Evaluate initial states
pBest = np.copy(X)
pBest_fitness = np.array([evaluate_fitness(p) for p in pBest])

gBest_idx = np.argmin(pBest_fitness)
gBest = np.copy(pBest[gBest_idx])
gBest_fitness = pBest_fitness[gBest_idx]

# 4. OPTIMIZATION LOOP
for iteration in range(MAX_ITERATIONS):
    for i in range(SWARM_SIZE):
        # Generate random stochastic vectors
        r1 = np.random.rand(NUM_TASKS)
        r2 = np.random.rand(NUM_TASKS)

        # Calculate new velocity & update position
        V[i] = W * V[i] + C1 * r1 * (pBest[i] - X[i]) + C2 * r2 * (gBest - X[i])
        X[i] = X[i] + V[i]

        # Boundary enforcement: Keep allocations within individual task boundaries
        X[i] = np.clip(X[i], MIN_RES, MAX_RES)

        # Proportional Normalization (Ensures total strictly matches capacity if needed)
        # Uncomment below if your system demands exactly Total_Capacity to be spent
        # X[i] = (X[i] / np.sum(X[i])) * TOTAL_CAPACITY

        # Evaluate fitness
        current_fitness = evaluate_fitness(X[i])

        # Update Personal Best
        if current_fitness < pBest_fitness[i]:
            pBest[i] = np.copy(X[i])
            pBest_fitness[i] = current_fitness

            # Update Global Best
            if current_fitness < gBest_fitness:
                gBest = np.copy(X[i])
                gBest_fitness = current_fitness

# 5. DISPLAY RESULTS
print("=== PSO Resource Allocation Complete ===")
for task_id, res in enumerate(gBest):
    print(f"Task {task_id + 1}: Allocated = {res:.2f} units (Ideal Target: {IDEAL_DEMAND[task_id]})")
print(f"Total Allocated: {np.sum(gBest):.2f} / {TOTAL_CAPACITY}")
print(f"Final Optimization Penalty Score: {gBest_fitness:.4f}")


=== PSO Resource Allocation Complete ===
Task 1: Allocated = 6.63 units (Ideal Target: 12)
Task 2: Allocated = 40.00 units (Ideal Target: 45)
Task 3: Allocated = 25.66 units (Ideal Target: 30)
Task 4: Allocated = 6.14 units (Ideal Target: 15)
Task 5: Allocated = 21.57 units (Ideal Target: 25)
Total Allocated: 100.00 / 100.0
Final Optimization Penalty Score: 263.8442
